In [1]:
# =============================================================================
# CÉLULA 1 — Imports
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# =============================================================================
# CÉLULA 2 — Carregar dados históricos
# =============================================================================
df_hist = spark.sql("""
    SELECT
        datetime,
        avg_temperature AS temp_int,
        avg_humidity    AS hum_int,
        temperature_ext,
        humidity_ext
    FROM gold_hourly_weather
    WHERE temperature_ext IS NOT NULL
      AND humidity_ext    IS NOT NULL
      AND avg_temperature IS NOT NULL
      AND avg_humidity    IS NOT NULL
    ORDER BY datetime
""").toPandas()

df_hist['datetime'] = pd.to_datetime(df_hist['datetime'])
df_hist = df_hist.sort_values('datetime').reset_index(drop=True)

print(f"Registos históricos: {len(df_hist)}")
print(f"Período: {df_hist['datetime'].min()} → {df_hist['datetime'].max()}")

# =============================================================================
# CÉLULA 3 — Features com lag + hora do dia
# =============================================================================
df_model = df_hist.copy()
df_model['temp_int_lag1'] = df_model['temp_int'].shift(1)
df_model['hum_int_lag1']  = df_model['hum_int'].shift(1)
df_model['hour']          = df_model['datetime'].dt.hour
df_model = df_model.dropna().reset_index(drop=True)

features = ['temperature_ext', 'humidity_ext', 'hour', 'temp_int_lag1', 'hum_int_lag1']
X      = df_model[features]
y_temp = df_model['temp_int']
y_hum  = df_model['hum_int']

# =============================================================================
# CÉLULA 4 — Treinar modelos
# =============================================================================
model_temp = RandomForestRegressor(n_estimators=100, random_state=42)
model_hum  = RandomForestRegressor(n_estimators=100, random_state=42)

model_temp.fit(X, y_temp)
model_hum.fit(X, y_hum)

print(f"Temperatura — R²:  {r2_score(y_temp, model_temp.predict(X)):.4f}")
print(f"Temperatura — MAE: {mean_absolute_error(y_temp, model_temp.predict(X)):.4f}")
print(f"Humidade    — R²:  {r2_score(y_hum, model_hum.predict(X)):.4f}")
print(f"Humidade    — MAE: {mean_absolute_error(y_hum, model_hum.predict(X)):.4f}")

# =============================================================================
# CÉLULA 5 — Carregar forecast
# =============================================================================
df_forecast = spark.sql("""
    SELECT
        event_timestamp,
        temperature_ext,
        humidity_ext
    FROM gold_weather_forecast
    ORDER BY event_timestamp
""").toPandas()

df_forecast['event_timestamp'] = pd.to_datetime(df_forecast['event_timestamp'])
df_forecast = df_forecast.sort_values('event_timestamp').reset_index(drop=True)

print(f"Registos de forecast: {len(df_forecast)}")
print(f"Período: {df_forecast['event_timestamp'].min()} → {df_forecast['event_timestamp'].max()}")

# =============================================================================
# CÉLULA 6 — Previsão recursiva com intervalo de confiança
# =============================================================================
now = datetime.now()

last      = df_hist.iloc[-1]
temp_prev = last['temp_int']
hum_prev  = last['hum_int']

results = []
for _, row in df_forecast.iterrows():
    hour = row['event_timestamp'].hour

    X_pred = pd.DataFrame(
        [[row['temperature_ext'], row['humidity_ext'], hour, temp_prev, hum_prev]],
        columns=features
    )

    # Previsão de cada árvore individualmente
    tree_preds_temp = np.array([tree.predict(X_pred.values)[0] for tree in model_temp.estimators_])
    tree_preds_hum  = np.array([tree.predict(X_pred.values)[0] for tree in model_hum.estimators_])

    temp_est = tree_preds_temp.mean()
    hum_est  = tree_preds_hum.mean()
    temp_min = tree_preds_temp.min()
    temp_max = tree_preds_temp.max()
    hum_min  = tree_preds_hum.min()
    hum_max  = tree_preds_hum.max()

    # Fator de incerteza — aumenta com a distância no tempo
    # Máximo de 7 dias (168h) → fator máximo de 2.0
    # Horas passadas (data já chegou) → fator mínimo de 1.0
    horas_distancia = max((row['event_timestamp'] - now).total_seconds() / 3600, 0)
    fator = 1 + (horas_distancia / 168)

    # Aplica o fator ao spread do intervalo
    temp_spread = (temp_max - temp_min) * fator / 2
    hum_spread  = (hum_max  - hum_min)  * fator / 2

    results.append({
        'datetime':                   row['event_timestamp'],
        'temperature_ext':            row['temperature_ext'],
        'humidity_ext':               row['humidity_ext'],
        'temp_interior_estimada':     round(temp_est, 2),
        'temp_interior_min':          round(temp_est - temp_spread, 2),
        'temp_interior_max':          round(temp_est + temp_spread, 2),
        'humidity_interior_estimada': round(hum_est, 2),
        'humidity_interior_min':      round(hum_est - hum_spread, 2),
        'humidity_interior_max':      round(hum_est + hum_spread, 2),
    })

    temp_prev = temp_est
    hum_prev  = hum_est

df_result = pd.DataFrame(results)
display(df_result)

# =============================================================================
# CÉLULA 7 — Guardar como tabela no lakehouse
# =============================================================================
spark.createDataFrame(df_result).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_interior_forecast")

print("Tabela gold_interior_forecast atualizada com sucesso.")

StatementMeta(, e0c54c9d-16ff-4861-9988-d9745609c8d1, 3, Finished, Available, Finished, False)

Registos históricos: 3398
Período: 2025-11-28 01:00:00 → 2026-04-22 16:00:00


Temperatura — R²:  0.9171
Temperatura — MAE: 0.0749
Humidade    — R²:  0.9834
Humidade    — MAE: 0.2698
Registos de forecast: 192
Período: 2026-04-22 00:00:00 → 2026-04-29 23:00:00


SynapseWidget(Synapse.DataFrame, 18dadd0a-599b-415f-a735-8aa3afddb080)

Tabela gold_interior_forecast atualizada com sucesso.
